# Fragrance Marketplace — Data Enrichment

This notebook enriches the cleaned marketplace dataset with additional analytical fields while preserving the original source values.

### Enrichment steps

1. **Product category**
   - Create a new `product_category` column based on the existing `type` and listing `title`.
   - Correct inconsistencies and missing values in the original `type`.
   - Categorize products into meaningful retail categories such as `edt`, `edp`, `parfum`, `extrait`, `set`, `lotion`, `body cream`, `aftershave`, `display`, `tester`, and other relevant categories identified in the data.

2. **Price standardization**
   - Extract the currency from `priceWithCurrency`.
   - Preserve the original `price` and `priceWithCurrency` values.
   - Create a standardized price field for cross-currency analysis.

3. **Gender enrichment**
   - Create a standardized gender-related field using the listing `title` and other available information.
   - Do not rely exclusively on the existing `perfumeType` field because it contains inconsistencies.

### Principles

- Keep the original marketplace fields unchanged.
- Create new enriched columns rather than overwriting source data.
- Base enrichment rules on the actual values in the dataset.
- Use explicit business rules for confirmed cases rather than broad assumptions.
- Preserve unusual but identifiable products such as sets, bundles, testers, display/factice bottles, and collectibles.
- Validate each enrichment step before moving to the next one.

In [0]:
from pyspark.sql import functions as F

# Load the globally cleaned marketplace dataset.
df = spark.table("main.fragrance_analytics.marketplace_clean")

# Display the schema and a sample of the cleaned data.
df.printSchema()
display(df.limit(20))

In [0]:
# Inspect existing product type values
type_summary = (
    df
    .groupBy("type")
    .agg(
        F.count("*").alias("listing_count")
    )
    .orderBy(F.desc("listing_count"))
)

display(type_summary)


# Inspect listings with missing product type
missing_type = (
    df
    .filter(F.col("type").isNull())
    .select(
        "id",
        "brand",
        "title",
        "type",
        "perfumeType",
        "price",
        "priceWithCurrency"
    )
    .orderBy("id")
)

print(f"Listings with missing type: {missing_type.count():,}")

display(missing_type)

In [0]:
# Product Enrichment — derive product category from title

from pyspark.sql import functions as F

# Load the cleaned marketplace dataset.
marketplace_clean = spark.table(
    "main.fragrance_analytics.marketplace_clean"
)

# Product category is derived from the listing title.
# Specific product forms are checked before generic fragrance terms.

marketplace_enriched = (
    marketplace_clean
    .withColumn(
        "product_category",
        F.when(
            # Gift sets, bundles and listings containing multiple products.
            F.col("title").rlike(
                r"(?i)("
                r"\bgift\s*set\b|"
                r"\bgiftset\b|"
                r"\bgift\s+box\b|"
                r"\bgift\s+bag\b|"
                r"\bperfume\s+set\b|"
                r"\bfragrance\s+set\b|"
                r"\bcollector\s+set\b|"
                r"\bset\s+(of|x)\b|"
                r"\bbundle\b|"
                r"\b\d+\s*(pc|pcs|piece|pieces)\s+set\b"
                r")"
            ),
            "set"
        )
        .when(
            # Display, factice, dummy and empty bottles.
            F.col("title").rlike(
                r"(?i)("
                r"\bfactice\b|"
                r"\bdisplay\s+bottle\b|"
                r"\bdisplay\s+only\b|"
                r"\bstore\s+display\b|"
                r"\bdummy\s+bottle\b|"
                r"\bdummy\s+display\b|"
                r"\bdisplay/factice\b|"
                r"\badvertising\s+display\b|"
                r"\bempty\s+bottle\b|"
                r"\bdisplay\b.*\b(bottle|music\s+box)\b"
                r")"
            ),
            "display"
        )
        .when(
            # Giant/oversized bottles that are clearly display-format items.
            F.col("title").rlike(
                r"(?i)("
                r"\bgiant\b.*\b\d+\"\b.*\b(bottle|flacon)\b|"
                r"\b\d+\"\b.*\b(giant|huge)\b.*\b(bottle|flacon)\b|"
                r"\b(giant|huge)\b.*\b(bottle|flacon)\b.*\b(display|limited)\b"
                r")"
            ),
            "display"
        )
        .when(
            # Testers are treated as their own product category.
            F.col("title").rlike(r"(?i)\btester\b"),
            "tester"
        )
        .when(
            # Advent calendars are kept separate from ordinary sets.
            F.col("title").rlike(r"(?i)\badvent\s+calendar\b"),
            "advent calendar"
        )
        .when(
            # Eau de Toilette variants.
            F.col("title").rlike(
                r"(?i)\b(eau\s*de\s*toilette|edt)\b"
            ),
            "edt"
        )
        .when(
            # Eau de Parfum spelling variants.
            F.col("title").rlike(
                r"(?i)\b("
                r"eau\s*de\s*parfum|"
                r"eau\s*de\s*perfume|"
                r"eau\s*de\s*parfume|"
                r"edp"
                r")\b"
            ),
            "edp"
        )
        .when(
            # Cologne and Eau de Cologne variants.
            F.col("title").rlike(
                r"(?i)\b(cologne|eau\s*de\s*cologne|edc)\b"
            ),
            "cologne"
        )
        .when(
            # Mist is kept as a separate product category.
            F.col("title").rlike(r"(?i)\bmist\b"),
            "mist"
        )
        .when(
            # Other non-perfume fragrance/body-care products.
            F.col("title").rlike(
                r"(?i)\b(after\s*shave|aftershave)\b"
            ),
            "aftershave"
        )
        .when(
            F.col("title").rlike(
                r"(?i)\b(body\s+cream|body\s+creme)\b"
            ),
            "body cream"
        )
        .when(
            F.col("title").rlike(r"(?i)\blotion\b"),
            "lotion"
        )
        .when(
            F.col("title").rlike(r"(?i)\b(powder|talc)\b"),
            "powder"
        )
        .when(
            F.col("title").rlike(r"(?i)\bdeodorant\b"),
            "deodorant"
        )
        .when(
            F.col("title").rlike(r"(?i)\bshower\s+gel\b"),
            "shower gel"
        )
        .when(
            F.col("title").rlike(r"(?i)\bsoap\b"),
            "soap"
        )
        .when(
            F.col("title").rlike(r"(?i)\bbath\s+oil\b"),
            "bath oil"
        )
        .when(
            # Oil/attar must be checked before generic perfume/parfum.
            F.col("title").rlike(
                r"(?i)\b(perfume\s+oil|fragrance\s+oil|attar)\b"
            ),
            "oil / attar"
        )
        .when(
            F.col("title").rlike(r"(?i)\blinen\s+spray\b"),
            "linen spray"
        )
        .when(
            F.col("title").rlike(r"(?i)\bfixative\b"),
            "fixative"
        )
        .when(
            # Generic perfume/parfum terminology.
            # Extrait and generic perfume/parfum wording map to parfum.
            F.col("title").rlike(
                r"(?i)\b("
                r"extrait\s+de\s+parfum|"
                r"pure\s+parfum|"
                r"pure\s+perfume|"
                r"parfum|"
                r"parfums|"
                r"perfume|"
                r"perfumes"
                r")\b"
            ),
            "parfum"
        )
        .otherwise("other")
    )
)

display(
    marketplace_enriched
    .select(
        "id",
        "brand",
        "title",
        "product_category"
    )
    .orderBy("id")
)